# CSP Aplicado: Programación de Horarios Universitarios
## 5 materias, 3 profesores, 3 franjas horarias, 3 salones

**Objetivo**: Asignar a cada materia una franja horaria y un salón, respetando restricciones reales de una universidad.

Este notebook sigue la misma estructura que el tutorial del mapa de Australia, pero con un problema más realista y con restricciones variadas (no solo "≠").

### Contenido:
1. Formulación del problema como CSP
2. Grafo de restricciones
3. Backtracking básico
4. Backtracking + MRV + LCV
5. Forward Checking
6. AC-3
7. Min-Conflicts
8. Comparación y análisis

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from collections import deque
from itertools import product
import copy, random

## 1. Formulación del Problema

### El escenario

La Facultad de Ingeniería debe programar 5 materias del semestre:

| Materia | Profesor | Necesita |
|---------|----------|----------|
| IA (Inteligencia Artificial) | Prof. García | Sala de cómputo |
| BD (Bases de Datos) | Prof. García | Sala de cómputo |
| Cálculo | Prof. López | Cualquier salón |
| Física | Prof. López | Lab. de física |
| Redes | Prof. Martínez | Sala de cómputo |

**Franjas horarias**: Lunes 8am, Lunes 10am, Martes 8am

**Salones**: Aula 101, Lab Cómputo, Lab Física

### Restricciones:
1. **Profesor**: un profesor no puede dar dos materias en la misma franja
2. **Salón**: un salón no puede tener dos materias en la misma franja
3. **Recurso**: cada materia necesita un tipo de salón específico
4. **Preferencia**: Cálculo no debería ser Martes 8am (el profesor pidió no ir los martes)

### Formulación como CSP:
- **Variables**: las 5 materias
- **Dominio de cada variable**: todas las combinaciones (franja, salón) posibles
- **Restricciones**: las reglas anteriores

In [ ]:
# ================================================================
# DEFINICIÓN DEL CSP DE HORARIOS
# ================================================================

# Variables: las 5 materias
variables = ["IA", "BD", "Cálculo", "Física", "Redes"]

# Franjas horarias y salones disponibles
franjas = ["Lun-8am", "Lun-10am", "Mar-8am"]
salones = ["Aula-101", "Lab-Cómputo", "Lab-Física"]

# Profesor de cada materia
profesor = {
    "IA": "García",
    "BD": "García",
    "Cálculo": "López",
    "Física": "López",
    "Redes": "Martínez"
}

# Salón requerido por cada materia (None = cualquiera)
salon_requerido = {
    "IA": "Lab-Cómputo",
    "BD": "Lab-Cómputo",
    "Cálculo": None,          # cualquier salón
    "Física": "Lab-Física",
    "Redes": "Lab-Cómputo"
}

# Generar dominios: combinaciones (franja, salón) válidas para cada materia
# Restricción de recurso se aplica aquí (restricción unaria → reduce dominio)
domains = {}
for var in variables:
    if salon_requerido[var]:
        # Solo el salón requerido
        domains[var] = [(f, salon_requerido[var]) for f in franjas]
    else:
        # Cualquier combinación
        domains[var] = [(f, s) for f in franjas for s in salones]

print("=" * 60)
print("FORMULACIÓN DEL CSP DE HORARIOS")
print("=" * 60)
print(f"\nVariables: {variables}")
print(f"Franjas: {franjas}")
print(f"Salones: {salones}")
print(f"\nDominios (después de aplicar restricciones unarias de recurso):")
for var in variables:
    print(f"  {var} ({profesor[var]}, necesita: {salon_requerido[var] or 'cualquiera'}):")
    for d in domains[var]:
        print(f"    {d[0]} en {d[1]}")
    print(f"    → {len(domains[var])} opciones")

### 1.1 Las restricciones binarias

Las restricciones entre pares de materias son de dos tipos:

**Tipo 1 — Mismo profesor**: Si dos materias comparten profesor, no pueden estar en la misma franja horaria.
- IA y BD (ambas Prof. García)
- Cálculo y Física (ambas Prof. López)

**Tipo 2 — Mismo salón**: Si dos materias necesitan el mismo salón, no pueden estar en la misma franja.
- IA, BD y Redes (todas necesitan Lab-Cómputo)

**Tipo 3 — Preferencia**: Cálculo no en Martes 8am (ya aplicada como restricción unaria al dominio).

Nota: las restricciones unarias (tipo de salón, preferencias) ya las aplicamos al construir los dominios. Las binarias las verificamos durante la búsqueda.

In [ ]:
# ================================================================
# RESTRICCIONES BINARIAS
# ================================================================

def get_constraint_pairs():
    """
    Genera pares de materias que tienen restricciones entre sí,
    y la razón de la restricción.
    """
    pairs = []
    reasons = {}
    
    for i, v1 in enumerate(variables):
        for v2 in variables[i+1:]:
            has_constraint = False
            reason_list = []
            
            # ¿Mismo profesor?
            if profesor[v1] == profesor[v2]:
                has_constraint = True
                reason_list.append(f"mismo profesor ({profesor[v1]})")
            
            # ¿Mismo salón requerido? (y ambos requieren uno específico)
            if (salon_requerido[v1] and salon_requerido[v2] and 
                salon_requerido[v1] == salon_requerido[v2]):
                has_constraint = True
                reason_list.append(f"mismo salón ({salon_requerido[v1]})")
            
            if has_constraint:
                pairs.append((v1, v2))
                reasons[(v1, v2)] = reason_list
    
    return pairs, reasons

constraint_pairs, constraint_reasons = get_constraint_pairs()

# Construir vecinos
neighbors = {v: [] for v in variables}
for a, b in constraint_pairs:
    neighbors[a].append(b)
    neighbors[b].append(a)

print("Restricciones binarias:")
print("-" * 50)
for (a, b) in constraint_pairs:
    reasons = ", ".join(constraint_reasons[(a, b)])
    print(f"  {a} ↔ {b}: {reasons}")

print(f"\nTotal: {len(constraint_pairs)} pares con restricciones")
print(f"\nVecinos:")
for v in variables:
    print(f"  {v}: {neighbors[v]}")

In [ ]:
# ================================================================
# FUNCIÓN DE CONSISTENCIA
# ================================================================

def is_consistent(var, value, assignment):
    """
    Verifica si asignar var=value es consistente con la asignación actual.
    value es una tupla (franja, salón).
    
    Dos materias son inconsistentes si:
    1. Mismo profesor Y misma franja (profesor no puede estar en dos lugares)
    2. Mismo salón Y misma franja (salón no puede tener dos clases)
    """
    franja_var, salon_var = value
    
    for other_var, other_value in assignment.items():
        if other_var == var:
            continue
        
        franja_other, salon_other = other_value
        
        # ¿Misma franja?
        if franja_var == franja_other:
            # Verificar conflicto de profesor
            if profesor[var] == profesor[other_var]:
                return False  # Mismo profesor, misma franja
            
            # Verificar conflicto de salón
            if salon_var == salon_other:
                return False  # Mismo salón, misma franja
    
    return True

# Tests
print("Tests de consistencia:")
print("-" * 50)
a1 = {"IA": ("Lun-8am", "Lab-Cómputo")}

print(f"\nAsignación actual: IA → Lun-8am en Lab-Cómputo")

# BD mismo profesor, misma franja
print(f"  ¿BD=(Lun-8am, Lab-Cómputo)? {is_consistent('BD', ('Lun-8am', 'Lab-Cómputo'), a1)}")
print(f"    → No: mismo profesor (García) + mismo salón + misma franja")

# BD mismo profesor, diferente franja
print(f"  ¿BD=(Lun-10am, Lab-Cómputo)? {is_consistent('BD', ('Lun-10am', 'Lab-Cómputo'), a1)}")
print(f"    → Sí: diferente franja")

# Redes diferente profesor, mismo salón y franja
print(f"  ¿Redes=(Lun-8am, Lab-Cómputo)? {is_consistent('Redes', ('Lun-8am', 'Lab-Cómputo'), a1)}")
print(f"    → No: mismo salón + misma franja")

# Cálculo diferente profesor, diferente salón, misma franja
print(f"  ¿Cálculo=(Lun-8am, Aula-101)? {is_consistent('Cálculo', ('Lun-8am', 'Aula-101'), a1)}")
print(f"    → Sí: diferente profesor Y diferente salón")

### 1.2 Grafo de Restricciones

Los nodos son las materias y las aristas representan restricciones entre ellas. El grosor/color de la arista indica el tipo de restricción.

In [ ]:
def plot_schedule_graph(assignment=None, title=""):
    G = nx.Graph()
    G.add_nodes_from(variables)
    
    pos = {"IA":(0,2), "BD":(2,2), "Cálculo":(3,0), "Física":(1,0), "Redes":(1,1)}
    
    # Colores por profesor
    prof_colors = {"García":"#E24B4A", "López":"#378ADD", "Martínez":"#1D9E75"}
    node_colors = [prof_colors[profesor[v]] for v in variables]
    
    fig, ax = plt.subplots(figsize=(9,6))
    
    # Dibujar aristas con etiquetas
    for a, b in constraint_pairs:
        reasons = constraint_reasons[(a,b)]
        style = "solid" if "mismo profesor" in reasons[0] else "dashed"
        color = "#E24B4A" if "mismo profesor" in reasons[0] else "#EF9F27"
        nx.draw_networkx_edges(G, pos, edgelist=[(a,b)], style=style,
                              edge_color=color, width=2.5, ax=ax)
    
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2200,
                          edgecolors="#444", linewidths=2, ax=ax)
    
    # Labels con info
    labels = {}
    for v in variables:
        if assignment and v in assignment:
            f, s = assignment[v]
            labels[v] = f"{v}\n{f}\n{s}"
        else:
            labels[v] = f"{v}\n({profesor[v]})"
    
    nx.draw_networkx_labels(G, pos, labels, font_size=9, font_weight="bold", ax=ax)
    
    # Leyenda
    import matplotlib.patches as mpatches
    legend = [mpatches.Patch(color=c, label=f"Prof. {p}") for p, c in prof_colors.items()]
    legend.append(plt.Line2D([0],[0], color="#E24B4A", linewidth=2, label="Mismo profesor"))
    legend.append(plt.Line2D([0],[0], color="#EF9F27", linewidth=2, linestyle="--", label="Mismo salón"))
    ax.legend(handles=legend, loc="upper right", fontsize=9)
    
    ax.set_title(title or "Grafo de Restricciones — Horarios", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

plot_schedule_graph()

## 2. Backtracking Básico

Asigna materias en orden fijo, probando cada combinación (franja, salón) del dominio.

In [ ]:
def backtracking_basic(variables, domains, neighbors):
    nodes = [0]
    def backtrack(assignment):
        if len(assignment) == len(variables):
            return dict(assignment)
        var = next(v for v in variables if v not in assignment)
        for value in domains[var]:
            nodes[0] += 1
            if is_consistent(var, value, assignment):
                assignment[var] = value
                result = backtrack(assignment)
                if result is not None: return result
                del assignment[var]
        return None
    return backtrack({}), nodes[0]

sol1, n1 = backtracking_basic(variables, domains, neighbors)
print("Backtracking básico:")
print("-" * 50)
for v in variables:
    f, s = sol1[v]
    print(f"  {v:10s} → {f:10s} en {s} (Prof. {profesor[v]})")
print(f"\nNodos explorados: {n1}")
plot_schedule_graph(sol1, f"Backtracking básico ({n1} nodos)")

## 3. Backtracking + MRV + LCV

**MRV**: asignar primero la materia con menos opciones válidas.
**LCV**: probar primero la combinación (franja, salón) que deja más opciones a las demás materias.

In [ ]:
def backtracking_mrv_lcv(variables, domains, neighbors):
    nodes = [0]
    def backtrack(assignment):
        if len(assignment) == len(variables):
            return dict(assignment)
        # MRV
        unassigned = [v for v in variables if v not in assignment]
        var = min(unassigned, key=lambda v:
                  sum(1 for val in domains[v] if is_consistent(v, val, assignment)))
        # LCV
        def count_eliminations(val):
            count = 0
            for nb in neighbors[var]:
                if nb not in assignment:
                    for nb_val in domains[nb]:
                        # ¿val eliminaría nb_val?
                        f1, s1 = val
                        f2, s2 = nb_val
                        if f1 == f2 and (profesor[var]==profesor[nb] or s1==s2):
                            count += 1
            return count
        
        for value in sorted(domains[var], key=count_eliminations):
            nodes[0] += 1
            if is_consistent(var, value, assignment):
                assignment[var] = value
                result = backtrack(assignment)
                if result is not None: return result
                del assignment[var]
        return None
    return backtrack({}), nodes[0]

sol2, n2 = backtracking_mrv_lcv(variables, domains, neighbors)
print("Backtracking + MRV + LCV:")
print("-" * 50)
for v in variables:
    f, s = sol2[v]
    print(f"  {v:10s} → {f:10s} en {s} (Prof. {profesor[v]})")
print(f"\nNodos explorados: {n2} (vs {n1} sin heurísticas)")

## 4. Forward Checking

Al asignar una materia, eliminar combinaciones imposibles de los dominios de las materias restantes.

In [ ]:
def backtracking_fc(variables, domains, neighbors):
    nodes = [0]
    def would_conflict(var1, val1, var2, val2):
        f1, s1 = val1; f2, s2 = val2
        if f1 != f2: return False
        if profesor[var1] == profesor[var2]: return True
        if s1 == s2: return True
        return False
    
    def backtrack(assignment, curr_domains):
        if len(assignment) == len(variables):
            return dict(assignment)
        # MRV
        unassigned = [v for v in variables if v not in assignment]
        var = min(unassigned, key=lambda v: len(curr_domains[v]))
        
        for value in list(curr_domains[var]):
            nodes[0] += 1
            if is_consistent(var, value, assignment):
                assignment[var] = value
                # Forward check
                saved = {}
                ok = True
                for nb in neighbors[var]:
                    if nb not in assignment:
                        saved[nb] = list(curr_domains[nb])
                        curr_domains[nb] = [v for v in curr_domains[nb]
                                           if not would_conflict(var, value, nb, v)]
                        if not curr_domains[nb]:
                            ok = False; break
                if ok:
                    result = backtrack(assignment, curr_domains)
                    if result is not None: return result
                for nb, sv in saved.items():
                    curr_domains[nb] = sv
                del assignment[var]
        return None
    
    cd = {v: list(d) for v, d in domains.items()}
    return backtrack({}, cd), nodes[0]

sol3, n3 = backtracking_fc(variables, domains, neighbors)
print("Forward Checking + MRV:")
print("-" * 50)
for v in variables:
    f, s = sol3[v]
    print(f"  {v:10s} → {f:10s} en {s} (Prof. {profesor[v]})")
print(f"\nNodos explorados: {n3}")

## 5. AC-3 (Arc Consistency)

Para horarios, un arco materia_i → materia_j es consistente si para cada (franja, salón) de materia_i, existe al menos una (franja, salón) de materia_j que no genera conflicto.

In [ ]:
def ac3_schedule(variables, doms, neighbors):
    """AC-3 adaptado para el problema de horarios."""
    def would_conflict(var1, val1, var2, val2):
        f1, s1 = val1; f2, s2 = val2
        if f1 != f2: return False
        if profesor[var1] == profesor[var2]: return True
        if s1 == s2: return True
        return False
    
    def revise(xi, xj):
        revised = False
        for x in list(doms[xi]):
            # ¿Existe al menos un valor en Dj compatible con x?
            has_support = any(not would_conflict(xi, x, xj, y) for y in doms[xj])
            if not has_support:
                doms[xi].remove(x)
                revised = True
        return revised
    
    queue = deque()
    for a, b in constraint_pairs:
        queue.append((a, b)); queue.append((b, a))
    
    steps = 0
    while queue:
        xi, xj = queue.popleft(); steps += 1
        if revise(xi, xj):
            if not doms[xi]:
                return False, steps
            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True, steps

# Aplicar AC-3 como preprocesamiento
d_ac3 = {v: list(d) for v, d in domains.items()}
print("Dominios ANTES de AC-3:")
for v in variables:
    print(f"  {v}: {len(d_ac3[v])} opciones")

ok, steps = ac3_schedule(variables, d_ac3, neighbors)
print(f"\nAC-3: {steps} revisiones, resultado: {'OK' if ok else 'FALLO'}")

print(f"\nDominios DESPUÉS de AC-3:")
for v in variables:
    print(f"  {v}: {len(d_ac3[v])} opciones")
    for val in d_ac3[v]:
        print(f"    {val[0]} en {val[1]}")

removed = sum(len(domains[v]) - len(d_ac3[v]) for v in variables)
print(f"\n→ AC-3 eliminó {removed} opciones por propagación de restricciones")

## 6. Min-Conflicts

Empieza con una asignación aleatoria y mejora iterativamente, cambiando la materia más conflictiva al horario que minimice conflictos.

In [ ]:
def min_conflicts_schedule(max_steps=500):
    def count_conflicts(var, value, assignment):
        count = 0
        f1, s1 = value
        for other, oval in assignment.items():
            if other == var: continue
            f2, s2 = oval
            if f1 == f2:
                if profesor[var] == profesor[other]: count += 1
                if s1 == s2: count += 1
        return count
    
    # Asignación aleatoria
    assignment = {v: random.choice(domains[v]) for v in variables}
    
    for step in range(max_steps):
        # Variables con conflictos
        conflicted = [v for v in variables
                      if count_conflicts(v, assignment[v], assignment) > 0]
        if not conflicted:
            return assignment, step
        
        var = random.choice(conflicted)
        # Valor que minimiza conflictos
        best = min(domains[var], key=lambda val: count_conflicts(var, val, assignment))
        assignment[var] = best
    
    return None, max_steps

random.seed(42)
print("Min-Conflicts — 10 ejecuciones:")
print("-" * 50)
for i in range(10):
    sol_mc, steps = min_conflicts_schedule()
    if sol_mc:
        conflicts = sum(1 for v in variables
                       for n in neighbors[v]
                       if n in sol_mc and sol_mc[v][0]==sol_mc[n][0]
                       and (profesor[v]==profesor[n] or sol_mc[v][1]==sol_mc[n][1]))
        print(f"  Run {i+1}: solución en {steps:3d} pasos")
    else:
        print(f"  Run {i+1}: no encontró solución")

## 7. Comparación de Estrategias

In [ ]:
results = {}
_, n1 = backtracking_basic(variables, domains, neighbors)
results["Backtracking\nbásico"] = n1
_, n2 = backtracking_mrv_lcv(variables, domains, neighbors)
results["BT + MRV\n+ LCV"] = n2
_, n3 = backtracking_fc(variables, domains, neighbors)
results["Forward\nChecking"] = n3

fig, ax = plt.subplots(figsize=(9,5))
colors = ["#E24B4A","#EF9F27","#1D9E75"]
bars = ax.bar(results.keys(), results.values(), color=colors, edgecolor="#444", linewidth=0.5)
for bar, n in zip(bars, results.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            str(n), ha="center", va="bottom", fontweight="bold", fontsize=16)
ax.set_ylabel("Nodos explorados"); ax.set_ylim(0, max(results.values())*1.3)
ax.set_title("Comparación — Horarios Universitarios (5 materias)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

print("Resumen:")
for s,n in results.items():
    print(f"  {s.replace(chr(10),' '):25s}: {n} nodos")

## 8. Visualización del Horario Final

In [ ]:
def plot_schedule(assignment, title="Horario Final"):
    fig, ax = plt.subplots(figsize=(10, 5))
    
    franja_y = {"Lun-8am": 2, "Lun-10am": 1, "Mar-8am": 0}
    salon_x = {"Aula-101": 0, "Lab-Cómputo": 1, "Lab-Física": 2}
    prof_colors = {"García":"#E24B4A", "López":"#378ADD", "Martínez":"#1D9E75"}
    
    # Dibujar grid
    for i in range(4): ax.axhline(y=i-0.5, color="#DDD", linewidth=0.5)
    for i in range(4): ax.axvline(x=i-0.5, color="#DDD", linewidth=0.5)
    
    # Colocar materias
    for var, (franja, salon) in assignment.items():
        x = salon_x[salon]
        y = franja_y[franja]
        color = prof_colors[profesor[var]]
        rect = plt.Rectangle((x-0.45, y-0.4), 0.9, 0.8, 
                             facecolor=color, alpha=0.3, edgecolor=color, linewidth=2)
        ax.add_patch(rect)
        ax.text(x, y+0.1, var, ha="center", va="center", fontweight="bold", fontsize=12)
        ax.text(x, y-0.15, f"Prof. {profesor[var]}", ha="center", va="center", fontsize=8, color="#666")
    
    ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.5, 2.5)
    ax.set_xticks(range(3)); ax.set_xticklabels(list(salon_x.keys()), fontsize=11)
    ax.set_yticks(range(3)); ax.set_yticklabels(list(franja_y.keys()), fontsize=11)
    ax.set_xlabel("Salón", fontsize=12); ax.set_ylabel("Franja Horaria", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_aspect("equal")
    plt.tight_layout(); plt.show()

sol_final, _ = backtracking_fc(variables, domains, neighbors)
plot_schedule(sol_final)

# Verificar restricciones
print("Verificación de restricciones:")
print("-" * 50)
all_ok = True
for a, b in constraint_pairs:
    fa, sa = sol_final[a]; fb, sb = sol_final[b]
    if fa == fb:
        if profesor[a] == profesor[b]:
            print(f"  ✗ {a} y {b}: mismo profesor ({profesor[a]}) en {fa}")
            all_ok = False
        if sa == sb:
            print(f"  ✗ {a} y {b}: mismo salón ({sa}) en {fa}")
            all_ok = False
    else:
        print(f"  ✓ {a} y {b}: OK ({fa}≠{fb} o sin conflicto)")
print(f"\n{'✓ Todas las restricciones satisfechas' if all_ok else '✗ Hay violaciones'}")

## 9. Resumen

### Lo que aprendimos con este ejemplo:

| Concepto | En el mapa de Australia | En horarios universitarios |
|---|---|---|
| **Variables** | Regiones | Materias |
| **Dominios** | 3 colores | Combinaciones (franja, salón) |
| **Restricciones unarias** | — | Tipo de salón requerido |
| **Restricciones binarias** | Adyacencia (≠ color) | Mismo profesor o mismo salón en misma franja |
| **MRV** | Región más restringida (SA) | Materia con menos opciones (IA, BD, Redes) |
| **Forward Checking** | Eliminar color de vecinos | Eliminar (franja,salón) incompatibles |
| **AC-3** | Poca poda con 3 colores | Puede podar combinaciones imposibles |

### Extensiones posibles:
- Agregar más materias (10, 20, 50...) y medir la escalabilidad
- Restricciones de capacidad del salón
- Preferencias de estudiantes (soft constraints → optimización)
- Materias de 2 horas (restricciones de franjas consecutivas)
- Múltiples secciones de la misma materia